# Reddit Logs to YTMusic Playlists

## Setup

In [1]:
# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..'
# Path to .tsv reddit search logs filtered to just new entries
reddit_log_path = '..\\..\\reddit-scraper\\db'
search_db_path = '..\\..\\reddit-scraper\\logs'

In [2]:
import os
import glob
import time

import unicodedata
from datetime import date
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

### YTMusic API and Functions

In [12]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(lambda x: x[0]['id'])
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(os.path.join(ytmusic_header_path, 'headers_auth.json'))
yt_res_cache = {}
yt_unmatched_cache = {}

### String Helper Functions

In [13]:
def scrub_title(title):
    orig_title = title
    title = str(title).lower().strip()
    title = unicodedata.normalize('NFKD', title).encode('ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    title = title.replace('| ', '(')
    # remove stuff at end of title
    for k in [
         'official video', 'music video', 'live video','lyric video', 'cover)', 'video)', 'prod.', 'produced by', 
         'album stream', 'album review', 'album version', 'full album','produced by', 'npr music tiny desk concert',
         'anniversary expanded edition']:
        if k in title:
            new_t = title.split(k)[0]
            if len(new_t) > 5:
                title = title.split(k)[0]
    start_char = ['[', '('] 
    for k in [
        'official', 'unoffical', 'free', 'explicit', 'video', 'music', 'nsfw', 'original', 'lyric', 'studio', 'vinyl',
        'full', 'album)', 'audio)', 'cover)', 'convert', 'thissongissick', 'duploc', 'prod', 'leak', 'from', 'lofi hip', 
        'remaster', 'uncensored', '720p', '1080p', '320k' 'repackag', '19', '20', 'dir', 'quality upgrade', 'complete', 
        'with lyrics', 'visualizer', 'deluxe', 'anniversary edition']:
        for s in start_char:
            t = s + k
            if t in title:
                new_t = title.split(t)[0]
                if len(new_t) > 5:
                    title = title.split(t)[0]
    for s in start_char:
        if title.endswith(s):
            title = title[:-1]     
    # remove tokens from title
    for k in ['[hd]', '[hq]', 'hd', 'hq', '()', '[]', ' | ', '{}']:
        if k in title:
            title=title.replace(k, '')
    if title.endswith(' - '):
        title = title[0:-3]
    return title
    

def check_album_scrub_title(title):
    # check for album
    title = title.lower().strip()
    is_album = False
    for k in ['full album', '(album)', 'album stream']:
        if k in title:
            is_album = True

    title = scrub_title(title)
    return title, is_album

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

## Parse Reddit .tsv and query YTMusic for Match

* Now checks db tsv to see if ialready matched (basd on sub and url)
* Some reddit entries are albums, most are tracks
* lots of title 'scrubbing' before query to clean
* saves match and unmatched seperatly, caches query responses 
    * cache in above ytmusic api cell
* scores match using fuzz metrics
    * tries to automate passing macthes with score threshold


#### Last run 12/29/2022


In [1]:
# Load last years db
db_tsv_path = os.path.join(search_db_path, 'ytmusic', 'reddit_ytmusic_subreddit_db_2021.tsv')
db = pd.read_csv(db_tsv_path, sep='\t', index_col=0)
db['reddit_post_id'] = db.reddit_sub + '//' + db.reddit_source_url
print(f'Loaded {len(db)} entries from {db_tsv_path}')
prev_match_ids = frozenset(db['reddit_post_id'])


NameError: name 'search_db_path' is not defined

In [ ]:
reddit_subfolders = ['new_2022']
log_tsvs = []
for folder in reddit_subfolders:
    tsv_path = os.path.join(reddit_log_path, folder)
    log_tsvs += list(glob.glob(os.path.join(tsv_path, '*.tsv')))
log_tsvs = sorted(log_tsvs)
print(f'Found {len(log_tsvs)} reddit tsvs')

log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 2
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']

matched_entries = []
unmatched_entries = []
for i, tsv_file in enumerate(log_tsvs):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg = toks
    df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    if len(df) == 0:
        print(f'\n\nSkipping empty tsv: {tsv_file}')
        continue
    print(f'\n\n({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    assert set(df.columns) == set(expected_cols)

    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = check_album_scrub_title(entry.title)
        
        # If already in db get match from there
        post_id = f'{sub}//{entry.url}'
        if post_id in prev_match_ids:
            continue
        # Check cache for saved YTMusic query response or previous match failures        
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_album', ''))}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_title', ''))}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_post_id'] = post_id
                unmatch['reddit_sub_id'] = f'{sub}//{entry.url}'
                unmatch['reddit_aggregator'] = agg
                unmatch['manual_label'] = 'no-match'
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print(f'Skipping : {entry.title}')
                continue


            # Other ytmusic fields
            # match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_post_id'] = post_id
        match['reddit_sub_id'] = f'{sub}//{entry.url}'
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id
        match['manual_label'] = f'no-label_{date.today()}'

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Using manual grading to set thresholds
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 60:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 75:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {entry.title}.\n  Match Error: {e}')


        # Update match results
        matched_entries.append(match)

# Process Matched entries
match_df = pd.DataFrame(matched_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match duplicates')
print(f'\n\nSaving output tsvs to {search_db_path}')
match_file = os.path.join(search_db_path, 'ytmusic',
                          f'reddit_2022-new_ytmusic_scored_new_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch duplicates')
unmatch_file = os.path.join(
    search_db_path, 'ytmusic', f'reddit_2022-new_ytmusic_failed_new_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)


Found 90 reddit tsvs


(0/90)  Loaded 460 entries with 14 columns from 2000smusic_all
Error with Britney Spears - I'm Not A Girl, Not Yet A Woman (Official HD Video): 'NoneType' object has no attribute 'get'
Skipping match score for Britney Spears - I'm Not A Girl, Not Yet A Woman (Official HD Video).
  Match Error: 'ytmusic_key'


(1/90)  Loaded 566 entries with 14 columns from 2010smusic_all
Error with blink-182 - She's Out Of Her Mind [MUSIC VIDEO]: 'NoneType' object has no attribute 'get'
Skipping match score for blink-182 - She's Out Of Her Mind [MUSIC VIDEO].
  Match Error: 'ytmusic_key'
Skipping : Nurse With Wound ‎– Creakiness And Other Misdemeanours (2012) Full Album


(2/90)  Loaded 168 entries with 14 columns from 50sMusic_all
Error with 1957 HITS ARCHIVE: Be-Bop Baby - Ricky Nelson (hit single version): 'NoneType' object has no attribute 'get'
Skipping match score for 1957 HITS ARCHIVE: Be-Bop Baby - Ricky Nelson (hit single version).
  Match Error: 'ytmusic_key'


(3/90)  

## Manually grade using gsheet
https://docs.google.com/spreadsheets/d/1Z6X6rmOoLTqo3na_8Owrdf2O6FJ08n71buPhiP5uawA/edit#gid=2087986722


In [4]:
graded_tsv = os.path.join(search_db_path, 'ytmusic',
                    'reddit_2022-new_ytmusic_scored_new_matches_2022-12-30_graded.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t')
graded_matches = graded_matches.sort_values('reddit_sub')

passing = graded_matches.loc[graded_matches.manual_label == 'passed-match']
failing = graded_matches.loc[graded_matches.manual_label == 'failed-match']
print(f'{graded_matches.shape} shaped graded matches, {len(passing)} passing, {len(failing)} failing')

(14724, 24) shaped graded matches, 9141 passing, 5583 failing


## Subreddit playlists for passing tracks TODO impl the 'add to existing, otherwise make new pl' logic seen in the albums cell belpw...)

(same code for round 1 and 2, just clear completed [])

In [11]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists
import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')


HEADER_FILE='../headers_auth.json'
PLAYCOUNT_FILE='../playlists/_ytmusic_lastfm_match_id_map.tsv'
NOT_LIKE_PLAYLIST_TSV ='../playlists/_not_liked_tracks.tsv'
Y = YTMusicPlaylists(header=HEADER_FILE, playcount_map=PLAYCOUNT_FILE,  not_like_tsv=NOT_LIKE_PLAYLIST_TSV)


Using ytmusicapi version: 0.25.0
Using header file: ../headers_auth.json
Loaded 266580 playounts from 104345 tracks


In [16]:
N=5
LIMIT = 2000
MIN_N_LIKE = 10
passing_tracks = passing.loc[passing.is_album == False]
completed = []
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not
for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing tracks')
    vids = df.ytmusic_videoId.unique().tolist()
    vids = list(frozenset(vids) - Y.banned_vid_set)
    title=f'x_r.{sub}_tracks_radio'
    desc = f'Matched {len(vids)} tracks from r.{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r.{sub} tracks playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Fetch Just Saved Playlist
    Y.clean_up_radio_playlist(
        Y.playlist_get_info(pl_id), verbose=True, 
        move_like=True, min_num_like=MIN_N_LIKE,
        sleep=1, create_like_playlist=True, 
        remove_dislike=True, remove_not_like=True
    )
    Y.playcount_sort_playlist(Y.playlist_get_info(pl_id, use_cache=False), ignore_banned=True)
    completed.append(sub)
    



Generating r.2000smusic ytmusic playlist for 410 passing tracks
Saved 402 r.2000smusic tracks playlist with id: PLWptjpDqazOxZj-oYnrQ1-16QkxNudiBJ, waiting 5 seconds...
Sorting x_r.2000smusic_tracks (PLWptjpDqazOxZj-oYnrQ1-16QkxNudiBJ) with 402 tracks into like and indifferent
Created like playlist: x_r.2000smusic_trackslike (32) and indifferent playlist: x_r.2000smusic_tracksradio (368 entries), deleted original playlist: x_r.2000smusic_tracks (402 entries)

Generating r.2010smusic ytmusic playlist for 482 passing tracks
Saved 478 r.2010smusic tracks playlist with id: PLWptjpDqazOy4rSxaKvyJLJ1JlA57TsZz, waiting 5 seconds...
Sorting x_r.2010smusic_tracks (PLWptjpDqazOy4rSxaKvyJLJ1JlA57TsZz) with 478 tracks into like and indifferent
Created like playlist: x_r.2010smusic_trackslike (15) and indifferent playlist: x_r.2010smusic_tracksradio (460 entries), deleted original playlist: x_r.2010smusic_tracks (478 entries)

Generating r.50sMusic ytmusic playlist for 138 passing tracks
Saved 134

## Subreddit playlists for passing albums
(same code for round 1 and 2, just clear completed [])

In [14]:
SLEEP_TIME=5
LIMIT = 900
MIN_N_LIKE = 5
passing_tracks = passing.loc[passing.is_album == True]
completed= []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'hiphop', 'hiphop101', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic', 'triphop']

for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r.{sub} ytmusic playlist for {len(df)} passing albums')
    vids = set()
    for row in df.itertuples():
        print(f' Adding album: {row.ytmusic_album}')
        try:
            for track in  ytm.get_album(row.ytmusic_albumId).get('tracks', []):
                vids.add(track['videoId'])
        except Exception as e:
            print(row, sub, e)
    vids = list(vids)
        
    title=f'x_r.{sub}_albums'
    try:
        existing_pl = Y.query_by_title(title)
        print('Adding to existing playlist')
        ytm.add_playlist_items(existing_pl.playlistId, video_ids=vids)
    except:
        desc = f'Matched {len(vids)} albums from r.{sub} using filters: {df.reddit_aggregator.unique()}'
        pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
        print(f'Saved {len(vids)} r.{sub} albums playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset to merge with tracks playlist
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']

    if len(liked_tracks) < MIN_N_LIKE:
        print(f'Not enough LIKE tracks to split into new playlists: count = {len(liked_tracks)}')
        continue
    vids = liked_tracks.videoId.unique().tolist()
    
    like_title=f'{title}_like'
    try:
        existing_like_pl = Y.query_by_title(existing_like_pl)
        print('Adding to existing like playlist')
        ytm.add_playlist_items(existing_like_pl.playlistId, video_ids=vids)
    except:
        desc = f'Liked subset of {len(vids)} entries from: {desc}'
        liked_pl_id = ytm.create_playlist(title=like_title, description=desc, video_ids=vids, privacy_status='PRIVATE')
        print(f'Filtered {len(vids)} LIKE r.{sub} albums playlist with id: {liked_pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)
    completed.append(sub)




Generating r.2000smusic ytmusic playlist for 3 passing albums
 Adding album: Absolution
 Adding album: Let Go
 Adding album: Hybrid Theory
No playlist with title: x_r.2000smusic_albums
Saved 40 r.2000smusic albums playlist with id: PLWptjpDqazOxVbItM3ZbUW_wY13rwdamR, waiting 5 seconds...
Not enough LIKE tracks to split into new playlists: count = 2

Generating r.70sMusic ytmusic playlist for 1 passing albums
 Adding album: Aja
Saved 7 r.70sMusic albums playlist with id: PLWptjpDqazOwsgWeKxDzAJZoBEoBeDelD, waiting 5 seconds...

Generating r.DreamPop ytmusic playlist for 3 passing albums
 Adding album: Fever Dream
 Adding album: Awakening:Sleeping
 Adding album: Souvlaki
Saved 27 r.DreamPop albums playlist with id: PLWptjpDqazOyT_2Wjbsm0FqjtGxzCwhUy, waiting 5 seconds...

Generating r.ElectronicMusic ytmusic playlist for 4 passing albums
 Adding album: Alive: 2007 (Live)
 Adding album: Unfold
 Adding album: In Decay
 Adding album: Psyence Fiction
Adding to existing playlist
Saved 53 r.E